[![Roboflow Notebooks](https://media.roboflow.com/notebooks/template/bannertest2-2.png?ik-sdk-version=javascript-1.4.3&updatedAt=1672932710194)](https://github.com/roboflow/notebooks)

# How to Tune Parameters to your Tracker
In this notebook you will download MOT17 dataset, format it for using the Tuner class, and tune the hiperparameters of the tracker.
Finally evaluate the best parameters found over the evaluation set.  

## Install Trackers


In [1]:
!pip install trackers
!pip install trackers[tune]

zsh:1: no matches found: trackers[tune]


In [15]:
import os
import random
import shutil
import tempfile
from pathlib import Path

import numpy as np

## Download the dataset
`Trackers` library provides us with CLI and python code to download the main datasets used in Multi-Object Tracking

In [16]:
!trackers download --help

usage: trackers download [-h] [--list] [--split SPLIT] [--asset ASSET]
                         [-o OUTPUT] [--cache-dir CACHE_DIR]
                         [dataset]

Download tracking datasets from the official trackers bucket.

positional arguments:
  dataset               Dataset name (e.g. mot17, sportsmot).

options:
  -h, --help            show this help message and exit
  --list                List available datasets, splits, and asset types.
  --split SPLIT         Comma-separated splits to download (e.g.
                        train,val,test). If omitted, all available splits are
                        downloaded.
  --asset ASSET         Comma-separated assets to download:
                        annotations,frames,detections. If omitted, all
                        available assets are downloaded.
  -o OUTPUT, --output OUTPUT
                        Output directory (default: current directory).
  --cache-dir CACHE_DIR
                        Cache directory for downloaded

In [17]:
!trackers download mot17 --split train,val --asset detections,annotations

[download] mot17:train:detections
  using cached mot17-train-public-detections.zip
[extract] mot17:train:detections
[done] mot17:train:detections
[download] mot17:train:annotations
  using cached mot17-train-annotations.zip
[extract] mot17:train:annotations
[done] mot17:train:annotations
[download] mot17:val:detections
  using cached mot17-val-public-detections.zip
[extract] mot17:val:detections
[done] mot17:val:detections
[download] mot17:val:annotations
  using cached mot17-val-annotations.zip
[extract] mot17:val:annotations
[done] mot17:val:annotations


## Format dataset for tuning

Tuner class expects a directory for tracks and a directory for detections like:
```text
data/
├── gt/
│   ├── MOT17-02-FRCNN
│   │   └── gt
│   │       └── gt.txt
│   │
│   ├── MOT17-04-FRCNN
│   └── ...
└── detections/
    ├── MOT17-02-FRCNN.txt
    ├── MOT17-04-FRCNN.txt
    └── ...
```
So lets format the downloaded dataset to fit this requirement

In [ ]:
def flatten_asset(split_dir: Path, asset: str, output_dir_name: str) -> Path:
    out_dir = split_dir / output_dir_name
    out_dir.mkdir(parents=True, exist_ok=True)

    for seq_dir in sorted(p for p in split_dir.iterdir() if p.is_dir()):
        # Skip generated flattened directories when re-running cells.
        if seq_dir.name.endswith("_flattened"):
            continue

        src = seq_dir / asset / f"{asset}.txt"
        dst = out_dir / f"{seq_dir.name}.txt"
        if src.exists():
            shutil.copy(src, dst)
            print(f"Copied {asset.upper()} {src} -> {dst}")
        else:
            print(f"Warning: {asset.upper()} source not found: {src}")

    print(f"{asset.upper()} files flattening complete.")
    return out_dir


def build_seqmap(det_dir: Path, gt_dir: Path) -> tuple[Path, list[str]]:
    det_sequences = {p.stem for p in det_dir.glob("*.txt")}
    gt_sequences = {p.stem for p in gt_dir.glob("*.txt")}
    common_sequences = sorted(det_sequences & gt_sequences)

    with tempfile.NamedTemporaryFile(mode="w", delete=False, suffix=".txt") as f:
        for seq in common_sequences:
            f.write(seq + "\n")
        return Path(f.name), common_sequences


gt_base_dir = Path("mot17")
train_split_dir = gt_base_dir / "train"

train_det_dir = flatten_asset(train_split_dir, asset="det", output_dir_name="det_flattened")
train_gt_dir = flatten_asset(train_split_dir, asset="gt", output_dir_name="gt_flattened")

# Define variables for the tuner
det_dir = str(train_det_dir)
gt_dir_for_tuner = str(train_gt_dir)
print(f"det_dir set to: {det_dir}")
print(f"gt_dir_for_tuner set to: {gt_dir_for_tuner}")

Copied DET mot17/train/MOT17-02-DPM/det/det.txt -> mot17/train/det_flattened/MOT17-02-DPM.txt
Copied DET mot17/train/MOT17-02-FRCNN/det/det.txt -> mot17/train/det_flattened/MOT17-02-FRCNN.txt
Copied DET mot17/train/MOT17-02-SDP/det/det.txt -> mot17/train/det_flattened/MOT17-02-SDP.txt
Copied DET mot17/train/MOT17-04-DPM/det/det.txt -> mot17/train/det_flattened/MOT17-04-DPM.txt
Copied DET mot17/train/MOT17-04-FRCNN/det/det.txt -> mot17/train/det_flattened/MOT17-04-FRCNN.txt
Copied DET mot17/train/MOT17-04-SDP/det/det.txt -> mot17/train/det_flattened/MOT17-04-SDP.txt
Copied DET mot17/train/MOT17-05-DPM/det/det.txt -> mot17/train/det_flattened/MOT17-05-DPM.txt
Copied DET mot17/train/MOT17-05-FRCNN/det/det.txt -> mot17/train/det_flattened/MOT17-05-FRCNN.txt
Copied DET mot17/train/MOT17-05-SDP/det/det.txt -> mot17/train/det_flattened/MOT17-05-SDP.txt
Copied DET mot17/train/MOT17-09-DPM/det/det.txt -> mot17/train/det_flattened/MOT17-09-DPM.txt
Copied DET mot17/train/MOT17-09-FRCNN/det/det.tx

In [39]:
temp_seqmap_path, common_sequences = build_seqmap(
    det_dir=Path(det_dir),
    gt_dir=Path(gt_dir_for_tuner),
)
temp_seqmap_file = str(temp_seqmap_path)
print(f"Using {len(common_sequences)} shared sequences for tuning.")

Using 7 shared sequences for tuning.


## Tune your favorite tracker

In [40]:
from trackers.tune import Tuner

Here we define the tuner, we select:
 - The `SORT` tracker.
 - `gt_dir`: path to ground truth tracks.
 - `det_dir`: path to detections over which the tracker will act.
 -  `objective`: objective metric to maximize.
 - `metrics`: list of all the metrics we want to keep a track of.
 - `n_trials`: number of combinations of hyperparameters to try.  

In [ ]:
import inspect

sig = inspect.signature(Tuner.__init__).parameters

tuner_kwargs = dict(
    objective="HOTA",
    metrics=["HOTA"],
    n_trials=15,
    seqmap=temp_seqmap_file,
)

tuner = Tuner("sort", gt_dir_for_tuner, det_dir, **tuner_kwargs)

### Run the parameter tuning
This will take a while: the tuner is trying 15 different combinations of parameters, finding by itself which changes to make to the parameters in order to maximize our objetive metric

In [50]:
best_params = tuner.run()

[I 2026-05-06 11:29:26,767] A new study created in memory with name: trackers-tune-sort
[I 2026-05-06 11:29:28,928] Trial 0 finished with value: 0.5186650561265436 and parameters: {'lost_track_buffer': 69, 'track_activation_threshold': 0.7130570280582736, 'minimum_consecutive_frames': 2, 'minimum_iou_threshold': 0.23540528081006296}. Best is trial 0 with value: 0.5186650561265436.
[I 2026-05-06 11:29:30,971] Trial 1 finished with value: 0.5074865193306491 and parameters: {'lost_track_buffer': 60, 'track_activation_threshold': 0.41309168934204377, 'minimum_consecutive_frames': 1, 'minimum_iou_threshold': 0.47399154789630427}. Best is trial 0 with value: 0.5186650561265436.
[I 2026-05-06 11:29:33,060] Trial 2 finished with value: 0.5045530896178886 and parameters: {'lost_track_buffer': 59, 'track_activation_threshold': 0.3159869813866671, 'minimum_consecutive_frames': 4, 'minimum_iou_threshold': 0.47128276336992686}. Best is trial 0 with value: 0.5186650561265436.
[I 2026-05-06 11:29:35,

and the best parameters are ...

In [51]:
print(best_params)

{'lost_track_buffer': 66, 'track_activation_threshold': 0.8218442811600641, 'minimum_consecutive_frames': 1, 'minimum_iou_threshold': 0.11111383270042534}


## Evaluate over the other set

First we will import some utilities to evaluate.

In [52]:
from trackers.tune.tuner import _run_tracker_on_detections
from trackers.eval.evaluate import evaluate_mot_sequences
from trackers import SORTTracker

We will now flatten the validation detections to: `/content/mot17/val/det_flattened`


In [53]:
val_split_dir = gt_base_dir / "val"
val_det_dir = flatten_asset(val_split_dir, asset="det", output_dir_name="det_flattened")


Copied DET mot17/val/MOT17-02-DPM/det/det.txt -> mot17/val/det_flattened/MOT17-02-DPM.txt
Copied DET mot17/val/MOT17-02-FRCNN/det/det.txt -> mot17/val/det_flattened/MOT17-02-FRCNN.txt
Copied DET mot17/val/MOT17-02-SDP/det/det.txt -> mot17/val/det_flattened/MOT17-02-SDP.txt
Copied DET mot17/val/MOT17-04-DPM/det/det.txt -> mot17/val/det_flattened/MOT17-04-DPM.txt
Copied DET mot17/val/MOT17-04-FRCNN/det/det.txt -> mot17/val/det_flattened/MOT17-04-FRCNN.txt
Copied DET mot17/val/MOT17-04-SDP/det/det.txt -> mot17/val/det_flattened/MOT17-04-SDP.txt
Copied DET mot17/val/MOT17-05-DPM/det/det.txt -> mot17/val/det_flattened/MOT17-05-DPM.txt
Copied DET mot17/val/MOT17-05-FRCNN/det/det.txt -> mot17/val/det_flattened/MOT17-05-FRCNN.txt
Copied DET mot17/val/MOT17-05-SDP/det/det.txt -> mot17/val/det_flattened/MOT17-05-SDP.txt
Copied DET mot17/val/MOT17-09-DPM/det/det.txt -> mot17/val/det_flattened/MOT17-09-DPM.txt
Copied DET mot17/val/MOT17-09-FRCNN/det/det.txt -> mot17/val/det_flattened/MOT17-09-FRCN

And now we can run the tracking over the validation set and then see the metrics

In [54]:
val_gt_dir = gt_base_dir / "val"
val_det_dir = Path(val_det_dir)
sequences = sorted(p.stem for p in val_det_dir.glob("*.txt"))
eval_kwargs = dict(
    gt_dir=val_gt_dir,
    metrics=["CLEAR", "HOTA", "Identity"],
)


def run_eval(tracker):
    with tempfile.TemporaryDirectory() as tmp:
        pred_dir = Path(tmp)
        for seq in sequences:
            tracker.reset()
            _run_tracker_on_detections(tracker, val_det_dir / f"{seq}.txt", pred_dir / f"{seq}.txt")
        return evaluate_mot_sequences(tracker_dir=pred_dir, **eval_kwargs)


First we will evaluate with default parameters

In [55]:
result_default = run_eval(SORTTracker())
print(result_default.table(columns=["HOTA", "MOTA" , "IDF1"]))

Sequence                        HOTA    MOTA    IDF1
----------------------------------------------------
MOT17-02-FRCNN                33.742  30.030  36.215
MOT17-04-FRCNN                55.348  48.813  62.383
MOT17-05-FRCNN                46.600  50.640  58.909
MOT17-09-FRCNN                49.461  50.747  56.474
MOT17-10-FRCNN                49.367  51.528  54.207
MOT17-11-FRCNN                48.992  54.572  54.689
MOT17-13-FRCNN                54.663  55.672  65.016
----------------------------------------------------
COMBINED                      49.950  46.769  56.088


and now lets see how tuning parameters worked out

In [56]:
result_tuned = run_eval(SORTTracker(**best_params))

print(result_tuned.table(columns=["HOTA", "MOTA", "IDF1"]))

Sequence                        HOTA    MOTA    IDF1
----------------------------------------------------
MOT17-02-FRCNN                34.844  30.081  36.782
MOT17-04-FRCNN                55.246  49.045  62.426
MOT17-05-FRCNN                47.274  52.398  59.273
MOT17-09-FRCNN                50.062  51.372  56.973
MOT17-10-FRCNN                50.062  52.034  55.821
MOT17-11-FRCNN                49.522  55.014  55.485
MOT17-13-FRCNN                58.514  59.981  69.936
----------------------------------------------------
COMBINED                      50.499  47.371  56.850


As you can see, we maximized for HOTA and got +0.6% HOTA, 0.5% MOTA and +0.8% IDF1. And by increasing the number of trials we can have even bigger improvements!

Try it out and get your Tracker to the best performance!

**Note**: MOT17 provides detections coming from different object detectors, in this notebook we evaluate using the ones from FRCNN.